# 📘 선택 실험: 공개 임베딩 모델과 도메인 학습 모델 비교

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">임베딩 모델 비교</strong>는 같은 문서와 평가 질의로 모델별 검색 성능을 재는 일입니다. 먼저 우리 분야에서 좋은 공개 모델을 고르고, 직접 학습한 모델이 그 기준보다 얼마나 달라졌는지 확인합니다.</p></div>

<img src="images/40_model_comparison.png" width="1050" alt="같은 문서·평가 질의·검색 지표로 공개 모델과 도메인 학습 모델을 비교합니다.">

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">공개 모델의 특징·크기와 추가 학습 전 성능을 먼저 비교합니다. 그다음 교안 01의 쌍 학습 모델을 평가해, 자기 베이스와 가장 좋은 공개 모델에 대한 변화량을 계산합니다.</p></div>

교안 01의 모델 저장까지 마친 뒤 새 커널에서 시작합니다. `output/labor_tuned`가 필요하며 이 부록에서는 모델을 추가로 학습하지 않습니다. 공개 모델을 처음 사용할 때는 파일을 내려받으므로 RunPod GPU 환경에서 진행합니다.

## 1. 비교할 모델 이해하기

### 1-1. 출발점과 학습 배경

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">공개 모델은 이미 사전학습된 상태입니다. 여기서 <strong style="font-size:1.13em;color:#1f3a5f;">추가 학습 전</strong>은 이번 노무 쌍 데이터로 파인튜닝하지 않은 상태를 뜻합니다.</p></div>

| 모델 | 어떤 모델인가 | 파라미터 규모 | 입력 상한 | 임베딩 차원 |
|---|---|---:|---:|---:|
| `intfloat/multilingual-e5-small` | 다국어 검색용 E5. 교안 01의 베이스 | 약 118M | 512토큰 | 384 |
| `dragonkue/multilingual-e5-small-ko` | 같은 E5-small에 한국어 질의·문서 쌍을 추가 학습한 모델 | 약 118M | 512토큰 | 384 |
| `nlpai-lab/KURE-v1` | BGE-M3에 한국어 검색 데이터를 추가 학습한 모델 | 약 568M | 8,192토큰 | 1,024 |
| 교안 01의 쌍 학습 모델 | 베이스 E5-small에 노무 질의·정답 조문을 MNR로 학습한 모델 | 약 118M | 512토큰 | 384 |

`M`은 백만 개입니다. 입력 상한은 프리픽스·특수 토큰을 포함하며, 아래 코드에서 실제 객체의 사양을 다시 확인합니다.

### 1-2. 비교에서 살필 차이

- **베이스 E5-small:** 다국어 자료로 대조 학습한 검색 모델입니다. 쌍 학습 모델과 같은 구조·크기이므로, 두 모델의 차이로 이번 도메인 학습의 효과를 봅니다. [제작자 모델 카드](https://huggingface.co/intfloat/multilingual-e5-small), [설정](https://huggingface.co/intfloat/multilingual-e5-small/blob/main/config.json).
- **e5-small-ko:** 같은 베이스를 한국어 검색에 맞춘 공개 모델입니다. 한국어 질의·문서와 오답 후보를 이용해 학습한 모델을 그대로 쓰는 선택지와 비교합니다. [제작자 모델 카드](https://huggingface.co/dragonkue/multilingual-e5-small-ko), [설정](https://huggingface.co/dragonkue/multilingual-e5-small-ko/blob/main/config.json).
- **KURE-v1:** 한국어·영어를 지원하며, 한국어 질의·문서·어려운 오답 자료로 추가 학습했습니다. 모델과 벡터가 더 크고 입력 상한도 깁니다. [제작자 모델 카드](https://huggingface.co/nlpai-lab/KURE-v1), [설정](https://huggingface.co/nlpai-lab/KURE-v1/blob/main/config.json).

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">모델별 원래 입력 상한을 사용하므로 긴 조문이 잘리는 범위도 다릅니다. 이 부록은 실제 모델 선택을 위한 비교이며, 구조·학습 자료·입력 길이가 다른 모델 간 점수 차이를 노무 학습 하나의 효과로 해석하지 않습니다.</p></div>

### 1-3. 공개 검색 벤치마크 참고

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">nDCG@10</strong>은 관련 문서를 상위 10개 안의 높은 순위에 놓을수록 커지는 검색 지표입니다. 아래는 제작자가 <strong style="font-size:1.13em;color:#1f3a5f;">같은 한국어 검색 7개 과제</strong>로 평가한 공개 점수이며, 이 노트북에서 실행한 결과와 구분합니다(확인: 2026-11-04).</p></div>

| 모델 | 7개 과제 평균 nDCG@10 | AutoRAG | Ko-StrategyQA |
|---|---:|---:|---:|
| multilingual-e5-small | 0.670906 | 0.80068 | 0.75157 |
| multilingual-e5-small-ko | 0.688819 | 0.86184 | 0.76173 |

7개는 XPQA·PublicHealthQA·MIRACL·Ko-StrategyQA·Belebele·AutoRAG·MrTidy입니다. 평균 개선이 모든 과제의 개선을 뜻하지는 않습니다. [동일 표의 제작자 공개 결과](https://huggingface.co/dragonkue/multilingual-e5-small-ko#evaluation).

KURE-v1 모델 카드는 **8개 과제 평균 nDCG@10 0.69473**을 보고합니다. 위 7개 평균과 평가 구성이 달라 숫자만으로 순위를 매기지 않습니다. 최신 KURE 저장소의 MTEB(kor, v2) 9개 과제 결과도 별도 평가입니다. [KURE-v1 공개 결과](https://huggingface.co/nlpai-lab/KURE-v1#evaluation), [현재 평가 구성](https://github.com/nlpai-lab/KURE#evaluation).

세 공개 모델과 우리 학습 모델을 **동등한 벤치마크 조건**으로 비교하려면 MTEB 버전·한국어 과제·split·후보 문서·지표를 고정해 네 모델을 모두 평가해야 합니다. 우리 학습 모델의 공개 벤치마크 점수는 아직 측정하지 않았습니다. 아래 실습에서는 네 모델을 같은 노무 문서·검증 질의로 직접 비교합니다.

## 2. 같은 평가 조건과 선정 기준

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">검색 대상 조문·검증 질의·정답 ID·정규화·지표를 고정합니다. 입력 프리픽스는 각 제작자의 사용 방법을 따릅니다.</p></div>

| 모델 | 질의 앞 | 문서 앞 |
|---|---|---|
| 베이스·e5-small-ko·쌍 학습 모델 | `query: ` | `passage: ` |
| KURE-v1 | 없음 | 없음 |

이 검색기는 상위 5개 문서를 제공하므로 **Recall@5를 주지표**로 미리 정합니다. 동점이면 MRR이 높은 모델을 고르고, Recall@1도 함께 확인합니다. 모델 카드의 외부 벤치마크 점수 대신 이 노무 평가셋에서 직접 잰 점수로 선택합니다.

노무 검증셋은 학습 중 최적 체크포인트와 비교 모델을 고르는 데 사용합니다. 아래 점수는 **모델 선택용 검증 결과**이며, 선택을 마친 모델의 최종 성능은 별도의 새 테스트셋에서 확인해야 합니다.

In [ ]:
from pathlib import Path

import pandas as pd
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다 (지난 단원과 같은 코드)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 조문이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 이 단원의 임베딩 모델과 e5 계열 프리픽스(질의는 "query: ", 문서는 "passage: ")
MODEL_ID = "intfloat/multilingual-e5-small"
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "

In [ ]:
# 교안 01의 함수 원문입니다. 같은 질문·정답 ID 규약으로 평가합니다.
import json


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def encode_texts(model, texts, prefix):
    """텍스트 목록에 프리픽스를 붙여 정규화된 임베딩(numpy 배열)으로 만듭니다."""
    return model.encode(
        [prefix + text for text in texts],   # 프리픽스를 붙인 문장 목록
        normalize_embeddings=True,           # 길이 1로 정규화해 벡터끼리의 내적이 곧 코사인 유사도가 되게 합니다
        show_progress_bar=False,             # 진행 막대를 끕니다(평가 때마다 여러 번 불러 출력이 길어지지 않게)
    )


import numpy as np


def gold_ranks(model, corpus, evalset, query_prefix=QUERY_PREFIX, passage_prefix=PASSAGE_PREFIX):
    """평가 질의마다 정답 문서가 유사도 순으로 몇 위인지(1위부터) 담은 배열을 돌려줍니다."""
    doc_ids = [doc["doc_id"] for doc in corpus]
    doc_emb = encode_texts(model, [doc["text"] for doc in corpus], passage_prefix)
    # 학습·평가 파일은 같은 세 열을 쓰며, 검색 평가는 질의와 정답 ID를 사용합니다.
    query_emb = encode_texts(model, [row["anchor"] for row in evalset], query_prefix)

    # 정규화한 벡터끼리의 내적이 코사인 유사도입니다. sims는 (질의 수, 문서 수) 행렬입니다
    sims = query_emb @ doc_emb.T

    # 질의마다 유사도가 높은 순으로 문서 id를 늘어놓고, 정답 문서가 몇 번째인지 셉니다
    ranks = []
    for i, row in enumerate(evalset):
        ranked_ids = [doc_ids[j] for j in np.argsort(-sims[i])]
        ranks.append(ranked_ids.index(row["doc_id"]) + 1)
    return np.array(ranks)

In [ ]:
# 검증 쌍은 학습과 같은 세 열이며, 검색 평가에는 anchor와 doc_id를 사용합니다.
labor_corpus = load_jsonl(data_dir / "본실습" / "labor_corpus.jsonl")
labor_eval = load_jsonl(data_dir / "본실습" / "labor_eval.jsonl")
print("검색 조문:", len(labor_corpus), "| 검증 질의:", len(labor_eval))

# 검색기의 상위 5개 결과를 기준으로 삼고, 점수를 보기 전에 선정 기준을 고정합니다.
PRIMARY_METRIC = "Recall@5"
TIE_BREAK_METRIC = "MRR"  # Recall@5가 같을 때만 사용합니다.

## 3. 추가 학습 전 공개 모델 비교

### 3-1. 사양 확인과 검색 평가

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">공개 모델 세 개를 먼저 평가합니다. 한 번에 한 모델만 보관하고, 실제 파라미터 수·입력 상한·차원과 정답 순위를 남깁니다.</p></div>

In [ ]:
from sentence_transformers import SentenceTransformer

# 각 모델에 맞는 프리픽스로 같은 조문과 검증 질의를 검색합니다.
public_models = [
    ("베이스 모델", MODEL_ID, QUERY_PREFIX, PASSAGE_PREFIX),
    ("e5-small-ko", "dragonkue/multilingual-e5-small-ko", QUERY_PREFIX, PASSAGE_PREFIX),
    ("KURE-v1", "nlpai-lab/KURE-v1", "", ""),  # KURE는 프리픽스를 붙이지 않습니다.
]
public_ranks = {}
model_info_rows = []
for model_name, model_id, query_prefix, passage_prefix in public_models:
    model = SentenceTransformer(model_id, device=device)
    # 소개한 사양을 실제 로딩한 객체에서 읽어 대조합니다.
    model_info_rows.append({
        "모델": model_name,
        "파라미터(M)": sum(parameter.numel() for parameter in model.parameters()) / 1_000_000,  # 백만 개 단위
        "입력 상한(토큰)": model.max_seq_length,
        "임베딩 차원": model.get_embedding_dimension(),
    })
    # 같은 평가 질의의 정답 순위를 모델별로 다시 구합니다.
    public_ranks[model_name] = gold_ranks(
        model, labor_corpus, labor_eval,
        query_prefix=query_prefix,
        passage_prefix=passage_prefix,
    )
    # GPU 메모리에 큰 모델 여러 개를 동시에 남기지 않습니다.
    del model

In [5]:
# 모델 객체에서 읽은 값입니다. M 단위만 소수 둘째 자리로 표시합니다.
model_info_table = pd.DataFrame(model_info_rows).set_index("모델")
display(model_info_table.round({"파라미터(M)": 2}))

,파라미터(M),입력 상한(토큰),임베딩 차원
모델,,,
베이스 모델,117.65,512,384
e5-small-ko,117.65,512,384
KURE-v1,567.75,8192,1024


### 3-2. 좋은 공개 모델 선정

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">선정과 변화량 계산에는 반올림 전 점수를 사용하고, 화면에서만 소수 셋째 자리로 표시합니다. 학습 모델을 보기 전에 기준 공개 모델을 정합니다.</p></div>

In [6]:
# 교안 01과 같은 지표식을 쓰되, 선정·차이 계산을 위해 반올림 전 값을 보관합니다.
public_score_rows = []
for model_name, ranks in public_ranks.items():
    public_score_rows.append({
        "모델": model_name,
        "Recall@1": float(np.mean(ranks <= 1)),
        "Recall@5": float(np.mean(ranks <= 5)),
        "MRR": float(np.mean(1 / ranks)),
    })
public_score_table = pd.DataFrame(public_score_rows).set_index("모델")
display(public_score_table.round(3))

# Recall@5가 가장 높은 모델을 고르고, 같으면 MRR로 비교합니다.
best_public_name = public_score_table.sort_values(
    [PRIMARY_METRIC, TIE_BREAK_METRIC], ascending=False,  # 두 지표 모두 클수록 좋습니다.
).index[0]
print("선정 기준:", PRIMARY_METRIC, "| 동률 기준:", TIE_BREAK_METRIC)
print("선정한 공개 모델:", best_public_name)

,Recall@1,Recall@5,MRR
모델,,,
베이스 모델,0.603,0.780,0.680
e5-small-ko,0.631,0.893,0.749
KURE-v1,0.715,0.888,0.798


선정 기준: Recall@5 | 동률 기준: MRR
선정한 공개 모델: e5-small-ko


<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li>Recall@5 1위 모델과 Recall@1·MRR 1위 모델이 다를 수 있습니다. 여기서는 미리 정한 Recall@5 기준을 유지합니다.</li>
<li>이 선정에는 공개 모델 세 개만 참여합니다. 다음 절의 쌍 학습 모델 점수로 기준 모델을 바꾸지 않습니다.</li>
</ul></div>

## 4. 저장한 쌍 학습 모델 평가

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01이 저장한 <code style="font-size:0.95em;color:#345473;">labor_tuned</code>를 불러와 같은 자료로 평가합니다. 베이스와 구조는 같고, 이번 노무 쌍 학습을 거친 가중치를 사용합니다.</p></div>

In [7]:
# 저장된 전체 임베딩 모델을 읽습니다. 이 부록에서는 학습하지 않습니다.
tuned_name = "쌍 학습 모델"
tuned_model = SentenceTransformer(str(output_dir / "labor_tuned"), device=device)
tuned_ranks = gold_ranks(tuned_model, labor_corpus, labor_eval)
model_info_rows.append({
    "모델": tuned_name,
    "파라미터(M)": sum(parameter.numel() for parameter in tuned_model.parameters()) / 1_000_000,
    "입력 상한(토큰)": tuned_model.max_seq_length,
    "임베딩 차원": tuned_model.get_embedding_dimension(),
})
model_info_table = pd.DataFrame(model_info_rows).set_index("모델")
# 학습한 모델의 사양을 앞의 공개 모델 표와 연결해 확인합니다.
display(model_info_table.loc[[tuned_name]].round({"파라미터(M)": 2}))
del tuned_model

,파라미터(M),입력 상한(토큰),임베딩 차원
모델,,,
쌍 학습 모델,117.65,512,384


## 5. 기존 기준보다 얼마나 달라졌는지 비교

### 5-1. 네 모델의 성능

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">베이스와 비교하면 이번 쌍 학습의 변화, 선정한 공개 모델과 비교하면 공개 모델을 그대로 쓰는 선택지에 대한 추가 이득을 확인합니다.</p></div>

In [8]:
# 공개 모델과 같은 계산으로 학습 모델의 반올림 전 점수를 구합니다.
tuned_scores = pd.Series({
    "Recall@1": float(np.mean(tuned_ranks <= 1)),
    "Recall@5": float(np.mean(tuned_ranks <= 5)),
    "MRR": float(np.mean(1 / tuned_ranks)),
}, name=tuned_name)
score_table = pd.concat([public_score_table, tuned_scores.to_frame().T])
display(score_table.round(3))

,Recall@1,Recall@5,MRR
베이스 모델,0.603,0.780,0.680
e5-small-ko,0.631,0.893,0.749
KURE-v1,0.715,0.888,0.798
쌍 학습 모델,0.621,0.916,0.742


### 5-2. 베이스·선정 공개 모델 대비 변화량

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">Recall의 변화는 <strong style="font-size:1.13em;color:#1f3a5f;">퍼센트포인트(%p)</strong> 로 표시합니다. 예를 들어 0.80→0.85는 +5%p입니다. MRR은 평균 순위 점수이므로 원래 점수의 차이로 표시합니다.</p></div>

In [9]:
# 자기 베이스와 선정 공개 모델을 비교 기준으로 삼되, 같은 모델이면 한 번만 비교합니다.
reference_names = ["베이스 모델"]
if best_public_name not in reference_names:
    reference_names.append(best_public_name)

# 반올림 전 점수로 차이를 구합니다. 양수는 개선, 음수는 하락입니다.
delta_rows = []
for reference_name in reference_names:
    delta = score_table.loc[tuned_name] - score_table.loc[reference_name]
    delta_rows.append({
        "비교 기준": reference_name,
        "Recall@1 변화(%p)": delta["Recall@1"] * 100,  # 비율 차이를 퍼센트포인트로 바꿉니다.
        "Recall@5 변화(%p)": delta["Recall@5"] * 100,
        "MRR 변화": delta["MRR"],  # MRR은 원점수 차이를 유지합니다.
    })
delta_table = pd.DataFrame(delta_rows).set_index("비교 기준")
display(delta_table.round(3))

,Recall@1 변화(%p),Recall@5 변화(%p),MRR 변화
비교 기준,,,
베이스 모델,1.869,13.551,0.062
e5-small-ko,-0.935,2.336,-0.007


<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li>선정한 공개 모델 대비 Recall@5 변화가 양수이면 이번 평가셋에서 상위 5개 정답 검색이 늘었습니다. 0 이하이면 추가 학습이 그 공개 기준을 넘었다고 결론 내리지 않습니다. Recall@1·MRR의 하락도 함께 읽습니다.</li>
<li>최종 선택에는 크기·입력 상한·벡터 차원도 고려합니다. 이 실험에서 측정하지 않은 속도·비용의 우위를 단정하지 않으며, 다른 분야의 퇴행은 교안 01·02의 별도 평가로 확인합니다.</li>
</ul></div>